In [1]:
import csv
from pathlib import Path

with open('books.csv', newline='', encoding='utf-8') as csv_file:
    books = list(csv.DictReader(csv_file))

len(books)

7

In [2]:
import requests

books_dir = Path('books')
books_dir.mkdir(exist_ok=True)

for book in books:
    pdf_response = requests.get(book['pdf_url'], timeout=60)
    pdf_response.raise_for_status()
    pdf_file = books_dir / f"{book['title']}.pdf"
    pdf_file.write_bytes(pdf_response.content)
    print(f"Downloaded {pdf_file}")

Downloaded books/Think Python 2e.pdf
Downloaded books/Think DSP.pdf
Downloaded books/Think Complexity 2e.pdf
Downloaded books/Think Java 2e.pdf
Downloaded books/Physical Modeling in MATLAB.pdf
Downloaded books/Think OS.pdf
Downloaded books/Think C++.pdf


In [3]:
from pathlib import Path
from markitdown import MarkItDown

markitdown = MarkItDown()
books_dir = Path('books')
books_text_dir = Path('books_text')
books_text_dir.mkdir(exist_ok=True)

for pdf_file in books_dir.glob('*.pdf'):
    result = markitdown.convert(str(pdf_file))
    markdown_file = books_text_dir / f"{pdf_file.stem}.md"
    markdown_file.write_text(result.text_content, encoding='utf-8')
    print(f"Converted {pdf_file} to {markdown_file}")

Converted books/Think OS.pdf to books_text/Think OS.md
Converted books/Think Java 2e.pdf to books_text/Think Java 2e.md
Converted books/Physical Modeling in MATLAB.pdf to books_text/Physical Modeling in MATLAB.md
Converted books/Think Python 2e.pdf to books_text/Think Python 2e.md
Converted books/Think DSP.pdf to books_text/Think DSP.md
Converted books/Think Complexity 2e.pdf to books_text/Think Complexity 2e.md
Converted books/Think C++.pdf to books_text/Think C++.md


In [4]:
think_python_file = books_text_dir / 'Think Python 2e.md'
think_python_content = think_python_file.read_text(encoding='utf-8')
len(think_python_content.splitlines())

9703

In [5]:
documents = []

for markdown_file in books_text_dir.glob('*.md'):
    content = markdown_file.read_text(encoding='utf-8').splitlines()
    content = [line for line in content if line.strip()]
    doc = {
        'source': markdown_file.name,
        'content': content
    }
    documents.append(doc)

len(documents)

7

In [7]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=100, step=50)

think_python_chunks = [
    chunk for chunk in document_chunks
    if chunk['source'] == 'Think Python 2e.md'
]

len(think_python_chunks)

189

In [8]:
from minsearch import Index

index_documents = []

for chunk in document_chunks:
    doc = chunk.copy()
    doc['content'] = '\n'.join(chunk['content'])
    index_documents.append(doc)

index = Index(text_fields=['content'])
index.fit(index_documents)

len(index_documents)

988

In [9]:
results = index.search('python function definition', num_results=5)
results[0]['source']

'Physical Modeling in MATLAB.md'

In [10]:
import json
from openai import OpenAI

openai_client = OpenAI()

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return (
        response.output_text,
        response.usage.input_tokens,
        response.usage.output_tokens
    )

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    return answer, input_tokens, output_tokens

query = 'python function definition'
answer, input_tokens, output_tokens = rag(query)
print(answer)
print(f'Input tokens: {input_tokens}')
print(f'Output tokens: {output_tokens}')

In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Here's a general structure for defining a function:

```python
def function_name(parameters):
    # Code block that performs some operations
    return result  # Optional return statement
```

### Example:

```python
def add(a, b):
    result = a + b
    return result
```

In this example, `add` is a function that takes two parameters, `a` and `b`, adds them together, and returns the result. 

### Key Points:
- **Function Definition**: Use the `def` keyword.
- **Parameters/Arguments**: Define inputs in parentheses.
- **Return Statement**: Use `return` to send a value back to the caller.
- **Function Invocation**: Call the function by using its name followed by parentheses, e.g., `add(2, 3)`.

By using functions, you can manage complexity and avoid variable name collisions, as each function has its own local scope.
Input tokens: 7918
Output tokens: 219


In [11]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return (
        response.output_parsed,
        response.usage.input_tokens,
        response.usage.output_tokens
    )

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    return answer, input_tokens, output_tokens

query = 'python function definition'
structured_answer, structured_input_tokens, structured_output_tokens = rag(query)
print(structured_answer.model_dump_json(indent=2))
print(f'Input tokens: {structured_input_tokens}')
print(f'Output tokens: {structured_output_tokens}')
print(f'More input tokens than Q5: {structured_input_tokens - 7918}')

{
  "answer": "To define a function in Python, you use the `def` keyword followed by the function name and parentheses containing any parameters. Here’s a basic example:\n\n```python\ndef my_function(parameter1, parameter2):\n    # Function body\n    result = parameter1 + parameter2\n    return result\n```\n\n### Key Components of a Function Definition:\n1. **Function Name**: `my_function` is the name of the function.\n2. **Parameters**: `parameter1` and `parameter2` are input values to the function.\n3. **Function Body**: It contains the code that performs the operation. In this case, it adds the two parameters.\n4. **Return Statement**: `return result` sends the result back to whoever called the function.\n\n#### Example Call:\nTo call this function and print the result:\n```python\nresult = my_function(3, 5)\nprint(result)  # Output: 8\n```",
  "found_answer": true,
  "confidence": 0.9,
  "confidence_explanation": "The context provides a detailed explanation of function definitions 